In [2]:
# Search Query Spelling Corrector

import re
import pandas as pd
import numpy as np

from pypdf import PdfReader

# Step 1: Load the spelling-error corpus
pdf_file = "Birkbeck spelling error corpus.pdf"
reader = PdfReader(pdf_file)
corpus_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        corpus_text += page_text + "\n"
print("Spelling Error Corpus Loaded Successfully")

# Step 2: Build vocabulary of correctly spelled words

vocabulary = set()
# Dictionary:
# spelling error -> correct word
error_mapping = {}
current_correct_word = None
for line in corpus_text.splitlines():
    line = line.strip()
    if line == "":
        continue
    # Correctly spelled words start with $
    if line.startswith("$"):
        word = line[1:].strip().lower()

        # Some PDF extraction can join text together.
        # Extract the first valid alphabetic word.
        match = re.match(r"[a-zA-Z]+", word)
        if match:
            current_correct_word = match.group().lower()
            vocabulary.add(current_correct_word)
    else:
        # Words below a correct word are spelling errors
        if current_correct_word is not None:
            errors = re.findall(
                r"[a-zA-Z]+",
                line.lower()
            )
            for error in errors:
                # Do not make a word its own error
                if error != current_correct_word:
                    error_mapping[error] = current_correct_word

# Convert vocabulary into sorted list
vocabulary = sorted(vocabulary)
print("\nVocabulary Size:", len(vocabulary))

print("\nSample Correct Words:")
print(vocabulary[:20])

# Display some corpus error mappings

print("\nSample Spelling Error Mappings:")
count = 0

for error, correct in error_mapping.items():
    print(error, "->", correct)
    count += 1
    if count == 10:
        break

# Step 3: Accept a user search query
query = input("\nEnter your search query: ")

# Step 4: Tokenize the query
words = re.findall(
    r"[a-zA-Z]+",
    query.lower()
)

print("\nTokenized Query:")
print(words)

# Step 5: Identify words not present in vocabulary
incorrect_words = []

for word in words:
    if word not in vocabulary:
        # If it is a known spelling error,
        # it definitely needs correction.
        if word in error_mapping:
            incorrect_words.append(word)
        else:
            # Unknown/OOV word
            # It is not automatically considered incorrect.
            incorrect_words.append(word)
print("\nWords Not Present in Vocabulary:")
print(incorrect_words)


# Step 6: Calculate Edit Distance
def edit_distance(word1, word2):
    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = np.zeros(
        (rows, cols),
        dtype=int
    )
    # First column
    for i in range(rows):
        matrix[i][0] = i
    # First row
    for j in range(cols):
        matrix[0][j] = j
    # Calculate edit distance
    for i in range(1, rows):
        for j in range(1, cols):
            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1
            deletion = matrix[i - 1][j] + 1
            insertion = matrix[i][j - 1] + 1
            substitution = (
                matrix[i - 1][j - 1] + cost
            )
            matrix[i][j] = min(
                deletion,
                insertion,
                substitution
            )
    return matrix[rows - 1][cols - 1]

# Step 7: Select the closest matching word
def closest_word(word):
    best_word = word
    best_distance = float("inf")

    for candidate in vocabulary:
        # Ignore candidates with very different lengths
        if abs(len(word) - len(candidate)) > 2:
            continue
        distance = edit_distance(
            word,
            candidate
        )
        if distance < best_distance:
            best_distance = distance
            best_word = candidate
    return best_word, best_distance

# Correction function
def correct_word(word):
    # Case 1:
    # Word is already correctly spelled
    if word in vocabulary:

        return word, 0, "Correct Word"
    # Case 2:
    # Very short words such as:
    # a, I, is, am, it, in, to, etc.
    # should not be blindly corrected.
    if len(word) <= 2:
        return (
            word,
            0,
            "Short Word - Kept Unchanged"
        )
    # Case 3:
    # Word is a known spelling error
    # in the Birkbeck corpus
    if word in error_mapping:
        correct = error_mapping[word]
        distance = edit_distance(
            word,
            correct
        )
        return (
            correct,
            distance,
            "Corpus Correction"
        )
    # Case 4:
    # Unknown / OOV word
    candidate, distance = closest_word(word)
    # Only correct if the candidate
    # is extremely close
    if distance <= 1:
        return (
            candidate,
            distance,
            "Edit Distance Correction"
        )
    else:
        # Do not blindly change unknown words
        return (
            word,
            distance,
            "Unknown Word - Kept Unchanged"
        )

# Step 8: Display corrections and corrected query
corrected_words = []

print("\nSuggested Corrections:")
print("----------------------")

for word in words:
    corrected_word, distance, reason = correct_word(word)
    corrected_words.append(corrected_word)
    # Display only words that required checking
    if word != corrected_word:
        print(
            word,
            "->",
            corrected_word,
            "(Edit Distance:",
            distance,
            ")"
        )
    elif reason == "Unknown Word - Kept Unchanged":
        print(
            word,
            "->",
            word,
            "(Unknown word, kept unchanged)"
        )
# Create corrected query
corrected_query = " ".join(
    corrected_words
)
print("\nOriginal Query:")
print(query)

print("\nCorrected Query:")
print(corrected_query)

# Step 9: Test the system with multiple spelling errors
print("\n======================================")
print("Testing Multiple Spelling Errors")
print("======================================")

test_queries = [
    "this is a beatiful movie",
    "the grammer is very good",
    "this is an excellant movie",
    "the goverment made a decision",
    "I recieved the receipt",
    "this is a beautiful day"
]

for test_query in test_queries:
    test_words = re.findall(
        r"[a-zA-Z]+",
        test_query.lower()
    )

    corrected_words = []

    for word in test_words:
        corrected_word, distance, reason = (
            correct_word(word)
        )
        corrected_words.append(
            corrected_word
        )
    corrected_test_query = " ".join(
        corrected_words
    )

    print("\nOriginal Query:")
    print(test_query)

    print("Corrected Query:")
    print(corrected_test_query)

print("\nSpelling Correction System Completed Successfully!")

Spelling Error Corpus Loaded Successfully

Vocabulary Size: 1920

Sample Correct Words:
['a', 'abandon', 'abandoned', 'abandoning', 'abandons', 'aberration', 'about', 'absence', 'absorbed', 'absorption', 'abuts', 'acceptable', 'accessible', 'accession', 'accidentally', 'acclimatization', 'accommodate', 'accommodated', 'accommodates', 'accommodating']

Sample Spelling Error Mappings:
apenines -> apennines
appenines -> apennines
athenean -> athenian
atheneans -> athenians
bernouilli -> bernoulli
blitzkreig -> blitzkrieg
brasillian -> brazilian
britian -> britain
brittish -> british
ceasar -> caesar



Enter your search query:  this is a beatiful day



Tokenized Query:
['this', 'is', 'a', 'beatiful', 'day']

Words Not Present in Vocabulary:
['is', 'beatiful', 'day']

Suggested Corrections:
----------------------
beatiful -> beautiful (Edit Distance: 1 )
day -> day (Unknown word, kept unchanged)

Original Query:
this is a beatiful day

Corrected Query:
this is a beautiful day

Testing Multiple Spelling Errors

Original Query:
this is a beatiful movie
Corrected Query:
this is a beautiful movie

Original Query:
the grammer is very good
Corrected Query:
the grammar is very good

Original Query:
this is an excellant movie
Corrected Query:
this is an excellent movie

Original Query:
the goverment made a decision
Corrected Query:
the government made a decision

Original Query:
I recieved the receipt
Corrected Query:
i received the receipt

Original Query:
this is a beautiful day
Corrected Query:
this is a beautiful day

Spelling Correction System Completed Successfully!
